In [ ]:
# CELL 1: Mount Drive (Colab only — skip on Kaggle)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

In [ ]:
# CELL 2: Install and imports
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

matplotlib.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'savefig.facecolor': 'white'
})

# Output directory on Drive
SAVE_DIR = '/content/drive/MyDrive/avec2014_figures'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Figures will be saved to: {SAVE_DIR}')

In [ ]:
# CELL 3: Load all data files from Drive
FUQ_CSV      = '/content/drive/MyDrive/avec2014_fuq_results.csv'
GENDER_CSV   = '/content/drive/MyDrive/avec2014_gender.csv'
RACE_CSV     = '/content/drive/MyDrive/avec2014_race.csv'
OCC_CSV      = '/content/drive/MyDrive/avec2014_occlusion.csv'

fuq_df    = pd.read_csv(FUQ_CSV)
gender_df = pd.read_csv(GENDER_CSV)
race_df   = pd.read_csv(RACE_CSV)
occ_df    = pd.read_csv(OCC_CSV)

# Build lookup maps
age_map     = dict(zip(gender_df['filename'], gender_df['age_group']))
race_map    = dict(zip(race_df['filename'],   race_df['race']))
glasses_map = dict(zip(occ_df['filename'],    occ_df['has_glasses']))
beard_map   = dict(zip(occ_df['filename'],    occ_df['has_beard']))

# Add all demographic columns to fuq_df
fuq_df['age_group']   = fuq_df['stem'].map(age_map)
fuq_df['race']        = fuq_df['stem'].map(race_map)
fuq_df['has_glasses'] = fuq_df['stem'].map(glasses_map).fillna(0).astype(int)
fuq_df['has_beard']   = fuq_df['stem'].map(beard_map).fillna(0).astype(int)
fuq_df['subgroup']    = fuq_df['gender'] + '_' + fuq_df['age_group'].fillna('Unknown')
fuq_df['mpiw']        = fuq_df['hi'] - fuq_df['lo']
fuq_df['abs_error']   = (fuq_df['y_true'] - fuq_df['y_pred']).abs()

print(f'FUQ results: {len(fuq_df)} videos')
print(f'Columns: {fuq_df.columns.tolist()}')
print(f'\nGender: {fuq_df["gender"].value_counts().to_dict()}')
print(f'Age:    {fuq_df["age_group"].value_counts().to_dict()}')
print(f'Race:   {fuq_df["race"].value_counts().to_dict()}')
print(f'Glasses:{fuq_df["has_glasses"].value_counts().to_dict()}')
print(f'Beard:  {fuq_df["has_beard"].value_counts().to_dict()}')

In [ ]:
# CELL 4: Helper functions
def group_stats(df, col, val_map=None):
    """
    Compute PICP, MPIW, MAE and N for each unique value in col.
    val_map: optional dict to rename values for display.
    Returns sorted DataFrame.
    """
    rows = []
    for val in df[col].dropna().unique():
        g    = df[df[col] == val]
        label = val_map.get(val, str(val)) if val_map else str(val)
        rows.append({
            'label': label,
            'N':     len(g),
            'PICP':  g['covered'].mean(),
            'MPIW':  g['mpiw'].mean(),
            'MAE':   g['abs_error'].mean()
        })
    return pd.DataFrame(rows).sort_values('PICP')

TARGET = 0.90
BLUE   = '#378ADD'
PINK   = '#D4537E'
AMBER  = '#EF9F27'
RED    = '#E24B4A'
GREEN  = '#1D9E75'
GRAY   = '#888780'
TEAL   = '#1D9E75'

def bar_color(picp):
    if picp < 0.75:  return RED
    if picp < 0.88:  return AMBER
    return GREEN

print('Helper functions defined.')

In [ ]:
# CELL 5: Figure 1 — Scatter plot: predicted vs true BDI-II score
# Real per-video predictions from avec2014_fuq_results.csv

fig, ax = plt.subplots(figsize=(7, 6))

male   = fuq_df[fuq_df['gender'] == 'M']
female = fuq_df[fuq_df['gender'] == 'F']

ax.scatter(male['y_true'],   male['y_pred'],   color=BLUE, marker='o',
           s=60, alpha=0.85, label=f'Male (N={len(male)})', zorder=3)
ax.scatter(female['y_true'], female['y_pred'], color=PINK, marker='^',
           s=70, alpha=0.85, label=f'Female (N={len(female)})', zorder=3)

# Perfect prediction line
lim = [0, 45]
ax.plot(lim, lim, '--', color=GRAY, linewidth=1.2, alpha=0.6, label='perfect prediction', zorder=2)

# Annotate uncovered points
uncovered = fuq_df[~fuq_df['covered']]
ax.scatter(uncovered['y_true'], uncovered['y_pred'],
           s=120, facecolors='none', edgecolors=RED, linewidths=1.5,
           zorder=4, label=f'outside interval (N={len(uncovered)})')

ax.set_xlim(-1, 46)
ax.set_ylim(-1, 46)
ax.set_xlabel('True BDI-II score', fontsize=12)
ax.set_ylabel('Predicted BDI-II score (50th quantile)', fontsize=12)
ax.set_title('Predicted vs true depression score — AVEC 2014 test set', fontsize=13, pad=12)
ax.legend(fontsize=10, framealpha=0.9)

# Overall MAE annotation
mae  = fuq_df['abs_error'].mean()
corr = fuq_df['y_true'].corr(fuq_df['y_pred'])
ax.text(0.05, 0.95, f'MAE = {mae:.2f}\nr = {corr:.2f}',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig1_scatter_pred_vs_true.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')

In [ ]:
# CELL 6: Figure 2 — PICP by demographic group (all groups)

# Build all group stats
groups = []

# Gender
for g, lbl in [('F','Female'), ('M','Male')]:
    d = fuq_df[fuq_df['gender']==g]
    groups.append({'label': lbl, 'category': 'gender', 'N': len(d),
                   'PICP': d['covered'].mean(), 'MPIW': d['mpiw'].mean(), 'MAE': d['abs_error'].mean()})

# Age
for a, lbl in [('Young','Young'), ('Old','Old')]:
    d = fuq_df[fuq_df['age_group']==a]
    if len(d) > 0:
        groups.append({'label': lbl, 'category': 'age', 'N': len(d),
                       'PICP': d['covered'].mean(), 'MPIW': d['mpiw'].mean(), 'MAE': d['abs_error'].mean()})

# Gender x Age subgroups
for sg, lbl in [('F_Young','F-Young'),('F_Old','F-Old'),('M_Young','M-Young'),('M_Old','M-Old')]:
    d = fuq_df[fuq_df['subgroup']==sg]
    if len(d) > 0:
        groups.append({'label': lbl, 'category': 'gender x age', 'N': len(d),
                       'PICP': d['covered'].mean(), 'MPIW': d['mpiw'].mean(), 'MAE': d['abs_error'].mean()})

# Race
for r in fuq_df['race'].dropna().unique():
    d = fuq_df[fuq_df['race']==r]
    groups.append({'label': r.title(), 'category': 'race', 'N': len(d),
                   'PICP': d['covered'].mean(), 'MPIW': d['mpiw'].mean(), 'MAE': d['abs_error'].mean()})

# Occlusion
for col, lbl0, lbl1 in [('has_glasses','No Glasses','Glasses'),('has_beard','No Beard','Beard')]:
    for val, lbl in [(0, lbl0), (1, lbl1)]:
        d = fuq_df[fuq_df[col]==val]
        if len(d) > 0:
            groups.append({'label': lbl, 'category': 'occlusion', 'N': len(d),
                           'PICP': d['covered'].mean(), 'MPIW': d['mpiw'].mean(), 'MAE': d['abs_error'].mean()})

gdf = pd.DataFrame(groups).sort_values('PICP').reset_index(drop=True)

cat_colors = {
    'gender':      BLUE,
    'age':         AMBER,
    'gender x age': '#7F77DD',
    'race':        TEAL,
    'occlusion':   PINK
}

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(
    range(len(gdf)),
    gdf['PICP'],
    color=[cat_colors[c] for c in gdf['category']],
    alpha=0.85,
    edgecolor='white',
    linewidth=0.5
)

# Target line
ax.axhline(TARGET, color=RED, linestyle='--', linewidth=1.5, label=f'target = {TARGET}')

# Labels on bars
for i, (_, row) in enumerate(gdf.iterrows()):
    ax.text(i, row['PICP'] + 0.005, f'{row["PICP"]:.2f}\nN={row["N"]}',
            ha='center', va='bottom', fontsize=7.5, color='#333')

ax.set_xticks(range(len(gdf)))
ax.set_xticklabels(gdf['label'], rotation=40, ha='right', fontsize=9)
ax.set_ylabel('PICP', fontsize=12)
ax.set_ylim(0.3, 1.12)
ax.set_title('Prediction interval coverage probability (PICP) by demographic group', fontsize=13, pad=12)

legend_elements = [mpatches.Patch(color=v, label=k) for k, v in cat_colors.items()]
legend_elements.append(Line2D([0],[0], color=RED, linestyle='--', label='target 0.90'))
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')

plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig2_picp_by_group.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')

In [ ]:
# CELL 7: Figure 3 — PICP Gap by sensitive attribute (ranking)

gap_data = {
    'Gender\n(F vs M)':         abs(fuq_df[fuq_df.gender=='F']['covered'].mean() - fuq_df[fuq_df.gender=='M']['covered'].mean()),
    'Age\n(Young vs Old)':      abs(fuq_df[fuq_df.age_group=='Young']['covered'].mean() - fuq_df[fuq_df.age_group=='Old']['covered'].mean()),
    'Race\n(White vs Latino)':  abs(fuq_df[fuq_df.race=='white']['covered'].mean() - fuq_df[fuq_df.race=='latino hispanic']['covered'].mean()),
    'Glasses\n(No vs Yes)':     abs(fuq_df[fuq_df.has_glasses==0]['covered'].mean() - fuq_df[fuq_df.has_glasses==1]['covered'].mean()),
    'Beard\n(No vs Yes)':       abs(fuq_df[fuq_df.has_beard==0]['covered'].mean() - fuq_df[fuq_df.has_beard==1]['covered'].mean()),
}

labels = list(gap_data.keys())
values = list(gap_data.values())
sorted_idx = np.argsort(values)[::-1]
labels = [labels[i] for i in sorted_idx]
values = [values[i] for i in sorted_idx]
colors = [RED if v > 0.15 else AMBER if v > 0.05 else GREEN for v in values]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(range(len(labels)), values, color=colors, alpha=0.85, edgecolor='white')

for i, v in enumerate(values):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=10)

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('PICP gap (absolute difference between groups)', fontsize=11)
ax.set_title('Fairness gap by sensitive attribute — AVEC 2014 test set', fontsize=13, pad=12)
ax.set_xlim(0, 0.48)
ax.axvline(0.05, color=GRAY, linestyle=':', linewidth=1, alpha=0.6)

legend_elements = [
    mpatches.Patch(color=RED,   label='large gap (>0.15)'),
    mpatches.Patch(color=AMBER, label='moderate gap (0.05–0.15)'),
    mpatches.Patch(color=GREEN, label='small gap (<0.05)'),
]
ax.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig3_picp_gap_ranking.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')

In [ ]:
# CELL 8: Figure 4 — MAE by demographic group

fig, ax = plt.subplots(figsize=(12, 5))

mae_sorted = gdf.sort_values('MAE', ascending=False).reset_index(drop=True)
ax.bar(
    range(len(mae_sorted)),
    mae_sorted['MAE'],
    color=[cat_colors[c] for c in mae_sorted['category']],
    alpha=0.85,
    edgecolor='white'
)

# Overall MAE line
overall_mae = fuq_df['abs_error'].mean()
ax.axhline(overall_mae, color=GRAY, linestyle='--', linewidth=1.5, label=f'overall MAE = {overall_mae:.2f}')

for i, (_, row) in enumerate(mae_sorted.iterrows()):
    ax.text(i, row['MAE'] + 0.1, f'{row["MAE"]:.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(range(len(mae_sorted)))
ax.set_xticklabels(mae_sorted['label'], rotation=40, ha='right', fontsize=9)
ax.set_ylabel('MAE (BDI-II points)', fontsize=12)
ax.set_title('Mean absolute error (MAE) by demographic group', fontsize=13, pad=12)
ax.set_ylim(0, 14)

legend_elements = [mpatches.Patch(color=v, label=k) for k, v in cat_colors.items()]
legend_elements.append(Line2D([0],[0], color=GRAY, linestyle='--', label=f'overall MAE {overall_mae:.2f}'))
ax.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig4_mae_by_group.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')

In [ ]:
# CELL 9: Figure 5 — CQR vs FUQ comparison
# MPIW reduction and per-gender PICP

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Left: MPIW comparison
ax = axes[0]
methods = ['CQR', 'FUQ']
mpiws   = [36.53, fuq_df['mpiw'].mean()]
colors  = ['#B5D4F4', BLUE]
bars    = ax.bar(methods, mpiws, color=colors, width=0.4, edgecolor='white', alpha=0.9)
for bar, val in zip(bars, mpiws):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3, f'{val:.2f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
reduction = (mpiws[0] - mpiws[1]) / mpiws[0] * 100
ax.annotate(f'−{reduction:.0f}%', xy=(1, mpiws[1]), xytext=(1.3, (mpiws[0]+mpiws[1])/2),
            fontsize=11, color=RED, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))
ax.set_ylabel('Mean prediction interval width (MPIW)', fontsize=11)
ax.set_title('Interval width: CQR vs FUQ', fontsize=12)
ax.set_ylim(0, 44)

# Right: Per-gender PICP for CQR and FUQ
ax = axes[1]
genders  = ['Female', 'Male']
# CQR and FUQ have same PICP in your results (converged at iter 0)
picp_f   = fuq_df[fuq_df.gender=='F']['covered'].mean()
picp_m   = fuq_df[fuq_df.gender=='M']['covered'].mean()
cqr_vals = [picp_f, picp_m]  # same as FUQ since optimization converged at 0
fuq_vals = [picp_f, picp_m]

x    = np.arange(len(genders))
w    = 0.3
ax.bar(x - w/2, cqr_vals, w, label='CQR', color='#B5D4F4', edgecolor='white', alpha=0.9)
ax.bar(x + w/2, fuq_vals, w, label='FUQ', color=BLUE,      edgecolor='white', alpha=0.9)
ax.axhline(TARGET, color=RED, linestyle='--', linewidth=1.5, label=f'target {TARGET}')
for i, (cv, fv) in enumerate(zip(cqr_vals, fuq_vals)):
    ax.text(i-w/2, cv+0.005, f'{cv:.2f}', ha='center', va='bottom', fontsize=9)
    ax.text(i+w/2, fv+0.005, f'{fv:.2f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(genders, fontsize=11)
ax.set_ylabel('PICP', fontsize=11)
ax.set_ylim(0.6, 1.05)
ax.set_title('Per-gender PICP: CQR vs FUQ', fontsize=12)
ax.legend(fontsize=9)

plt.suptitle('CQR vs FUQ — AVEC 2014', fontsize=13, y=1.01)
plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig5_cqr_vs_fuq.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')

In [ ]:
# CELL 10: Figure 6 — Prediction intervals visualised
# Show intervals for each test video sorted by true score
# Covered = blue, uncovered = red

df_sorted = fuq_df.sort_values('y_true').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 6))

for i, row in df_sorted.iterrows():
    color = BLUE if row['covered'] else RED
    ax.plot([i, i], [row['lo'], row['hi']], color=color, alpha=0.5, linewidth=1.5)
    ax.plot(i, row['y_pred'], 'o', color=color, markersize=3, alpha=0.8)

ax.scatter(range(len(df_sorted)), df_sorted['y_true'],
           color='black', s=20, zorder=5, label='true score')

covered_patch   = mpatches.Patch(color=BLUE, alpha=0.6, label=f'covered interval (N={df_sorted["covered"].sum()})')
uncovered_patch = mpatches.Patch(color=RED,  alpha=0.6, label=f'uncovered interval (N={(~df_sorted["covered"]).sum()})')
true_scatter    = Line2D([0],[0], marker='o', color='w', markerfacecolor='black', markersize=6, label='true score')
ax.legend(handles=[covered_patch, uncovered_patch, true_scatter], fontsize=9)

ax.set_xlabel('Test videos (sorted by true BDI-II score)', fontsize=11)
ax.set_ylabel('BDI-II score', fontsize=11)
ax.set_title(f'FUQ prediction intervals — AVEC 2014 test set (PICP={df_sorted["covered"].mean():.2f})', fontsize=13, pad=12)
ax.set_xlim(-1, len(df_sorted))
ax.set_ylim(-3, 50)

plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig6_prediction_intervals.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')

In [ ]:
# CELL 11: Figure 7 — Gender x Age subgroup heatmap

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

subgroups = ['F_Young', 'F_Old', 'M_Young', 'M_Old']
labels    = ['F\nYoung', 'F\nOld', 'M\nYoung', 'M\nOld']
metrics   = ['PICP', 'MAE', 'MPIW']
titles    = ['PICP', 'MAE', 'MPIW']

results = []
for sg in subgroups:
    d = fuq_df[fuq_df['subgroup']==sg]
    results.append({
        'PICP': d['covered'].mean() if len(d)>0 else np.nan,
        'MAE':  d['abs_error'].mean() if len(d)>0 else np.nan,
        'MPIW': d['mpiw'].mean() if len(d)>0 else np.nan,
        'N':    len(d)
    })
res_df = pd.DataFrame(results, index=labels)

cmaps = ['RdYlGn', 'RdYlGn_r', 'RdYlGn_r']
for ax, metric, title, cmap in zip(axes, metrics, titles, cmaps):
    vals = res_df[metric].values.reshape(1, -1)
    im   = ax.imshow(vals, cmap=cmap, aspect='auto',
                     vmin=np.nanmin(vals)*0.9, vmax=np.nanmax(vals)*1.05)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_yticks([])
    ax.set_title(title, fontsize=12)
    for j, (val, sg) in enumerate(zip(res_df[metric], subgroups)):
        n = res_df.loc[labels[j], 'N']
        txt = f'{val:.2f}\nN={n}' if not np.isnan(val) else 'N/A'
        ax.text(j, 0, txt, ha='center', va='center', fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.6)

plt.suptitle('Gender × age subgroup analysis — FUQ results', fontsize=13, y=1.02)
plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig7_gender_age_heatmap.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')

In [ ]:
# CELL 12: Figure 8 — Race analysis

race_stats = group_stats(fuq_df, 'race')
race_stats = race_stats[race_stats['N'] >= 2]  # exclude tiny groups

fig, axes = plt.subplots(1, 3, figsize=(13, 5))

race_colors = {'white': GREEN, 'latino hispanic': RED, 'asian': BLUE}
colors = [race_colors.get(r.lower(), GRAY) for r in race_stats['label']]

for ax, metric, title, ylbl in zip(
    axes,
    ['PICP', 'MAE', 'MPIW'],
    ['PICP by race', 'MAE by race', 'MPIW by race'],
    ['PICP', 'MAE (BDI-II points)', 'MPIW (BDI-II points)']
):
    bars = ax.bar(range(len(race_stats)), race_stats[metric], color=colors, alpha=0.85, edgecolor='white')
    if metric == 'PICP':
        ax.axhline(TARGET, color=RED, linestyle='--', linewidth=1.5, alpha=0.7)
        ax.set_ylim(0.4, 1.15)
    for i, (_, row) in enumerate(race_stats.iterrows()):
        ax.text(i, row[metric]+0.005 if metric=='PICP' else row[metric]+0.1,
                f'{row[metric]:.2f}\nN={row["N"]}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(range(len(race_stats)))
    ax.set_xticklabels([r.title() for r in race_stats['label']], fontsize=9)
    ax.set_ylabel(ylbl, fontsize=10)
    ax.set_title(title, fontsize=11)

plt.suptitle('Race/ethnicity analysis — FUQ results (AVEC 2014 test set)', fontsize=13, y=1.01)
plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig8_race_analysis.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')

In [ ]:
# CELL 13: Figure 9 — Occlusion analysis

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

for row_idx, (col, lbl0, lbl1) in enumerate([('has_glasses','No Glasses','Glasses'),
                                              ('has_beard',  'No Beard',  'Beard')]):
    d0 = fuq_df[fuq_df[col]==0]
    d1 = fuq_df[fuq_df[col]==1]

    for col_idx, (metric, title, ylbl) in enumerate([
        ('covered', 'PICP', 'PICP'),
        ('abs_error', 'MAE', 'MAE (BDI-II points)'),
        ('mpiw', 'MPIW', 'MPIW (BDI-II points)')
    ]):
        ax  = axes[row_idx][col_idx]
        v0  = d0[metric].mean()
        v1  = d1[metric].mean() if len(d1) > 0 else 0
        c   = [GREEN, RED] if metric == 'covered' else [BLUE, AMBER]
        bars = ax.bar([f'{lbl0}\nN={len(d0)}', f'{lbl1}\nN={len(d1)}'], [v0, v1],
                      color=c, alpha=0.85, edgecolor='white', width=0.5)
        for bar, val in zip(bars, [v0, v1]):
            ax.text(bar.get_x()+bar.get_width()/2, val+0.005 if metric=='covered' else val+0.1,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        if metric == 'covered':
            ax.axhline(TARGET, color=RED, linestyle='--', linewidth=1.2, alpha=0.7)
            ax.set_ylim(0.3, 1.15)
        ax.set_ylabel(ylbl, fontsize=9)
        ax.set_title(f'{title} — {"glasses" if col=="has_glasses" else "beard"}', fontsize=10)

plt.suptitle('Facial occlusion analysis — FUQ results (AVEC 2014 test set)', fontsize=13, y=1.01)
plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig9_occlusion_analysis.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')

In [ ]:
# CELL 14: Figure 10 — Error distribution by gender
# Violin + box plot showing spread of errors

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: error distribution by gender
ax = axes[0]
male_errors   = fuq_df[fuq_df.gender=='M']['abs_error'].values
female_errors = fuq_df[fuq_df.gender=='F']['abs_error'].values
vp = ax.violinplot([male_errors, female_errors], positions=[1,2],
                   showmedians=True, showextrema=True)
for pc, c in zip(vp['bodies'], [BLUE, PINK]):
    pc.set_facecolor(c)
    pc.set_alpha(0.6)
ax.set_xticks([1,2])
ax.set_xticklabels([f'Male\nN={len(male_errors)}', f'Female\nN={len(female_errors)}'], fontsize=11)
ax.set_ylabel('Absolute error (BDI-II points)', fontsize=11)
ax.set_title('Error distribution by gender', fontsize=12)

# Right: error distribution by race
ax = axes[1]
race_groups = [(r, fuq_df[fuq_df.race==r]['abs_error'].values)
               for r in ['white', 'latino hispanic', 'asian']
               if len(fuq_df[fuq_df.race==r]) >= 2]
positions   = list(range(1, len(race_groups)+1))
race_colors_list = [GREEN, RED, BLUE]
vp = ax.violinplot([g[1] for g in race_groups], positions=positions,
                   showmedians=True, showextrema=True)
for pc, c in zip(vp['bodies'], race_colors_list):
    pc.set_facecolor(c)
    pc.set_alpha(0.6)
ax.set_xticks(positions)
ax.set_xticklabels([f'{r.title()}\nN={len(g)}' for r, g in race_groups], fontsize=9)
ax.set_ylabel('Absolute error (BDI-II points)', fontsize=11)
ax.set_title('Error distribution by race', fontsize=12)

plt.suptitle('Prediction error distributions', fontsize=13, y=1.01)
plt.tight_layout()
path = os.path.join(SAVE_DIR, 'fig10_error_distributions.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')

In [ ]:
# CELL 15: Summary — list all saved figures
print('=== All figures saved to Drive ===')
for f in sorted(os.listdir(SAVE_DIR)):
    if f.endswith('.png'):
        size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e3
        print(f'  {f}  ({size:.0f} KB)')